# M5.1 Current Flow in Conducting Sheets

Book pointer: chapter 9, sections 9.2 (Ohm's law on microscopic form), 9.3 (resistance), 9.4 (energy loss), 9.5 (conservation of charge), and chapter 10, section 10.3 (circuits as models).

## 1. Recall

Answer on paper before running any cell.

1. Write Ohm's law in microscopic form. Which quantity is a property of the material, and which of the geometry?
2. Derive $R = L/(\sigma A)$ for a uniform bar from $\mathbf{J} = \sigma\mathbf{E}$ and the definitions of current and voltage.
3. In a steady state, $\nabla\cdot\mathbf{J} = 0$. What does that say about the currents meeting at one node of a resistor network?
4. At the edge of a conductor that borders an insulator, which component of $\mathbf{J}$ must vanish, and what does that imply for $\partial V/\partial n$?
5. Write the power dissipated per unit volume in terms of $\mathbf{J}$ and $\mathbf{E}$, and in terms of $\mathbf{J}$ and $\sigma$.

## 2. Principle to algorithm

A steady current has no sources or sinks inside the conductor, so $\nabla\cdot\mathbf{J} = 0$. With $\mathbf{J} = \sigma\mathbf{E}$ and $\mathbf{E} = -\nabla V$ this becomes

$$\nabla\cdot(\sigma\nabla V) = 0.$$

That is the equation of the dielectric solver in module 3, $\nabla\cdot(\epsilon\nabla V) = -\rho_f$, with $\sigma$ in place of $\epsilon$ and no source. The same discretisation works. Put a node at every grid point. Each face between two neighbouring nodes is a resistor. For a sheet of thickness $t$ and grid spacing $h$, a face has length $h$ and cross-section $h\,t$, so its conductance is $\sigma_{\text{face}}\,h\,t/h = \sigma_{\text{face}}\,t$. At a change of material the face conductivity is the harmonic mean of the two node values, which is the series combination of two half-cells and keeps the normal current continuous. Kirchhoff's current law at every free node is then the discrete $\nabla\cdot\mathbf{J} = 0$.

So a conducting sheet is a resistor network on a grid, which is the point of section 10.3: a circuit is a model of a continuous conductor, and a continuous conductor can be computed as a circuit. `emlib.circuits.solve_sheet` builds the network with `grid_edges`, holds the left column of nodes at $V_{\text{left}}$ and the right column at $V_{\text{right}}$ (the electrodes), and solves it with `solve_network`. It returns the node potentials, the current through every face, and the total current.

Insulating edges need no special treatment. The top and bottom rows of nodes simply have no faces beyond them, so no current can leave there, which is the condition $J_n = 0$. A hole is a region with $\sigma = 0$: the harmonic mean of anything with zero is zero, so the faces into the hole carry nothing.

Geometry convention. Each node owns a cell of width $h$ centred on it, so a sheet of $n_x\times n_y$ nodes has width $W = n_y h$ (the insulating edges lie half a cell outside the outer node rows) and length $L = (n_x - 1)h$ between the electrode columns.

## 3. Build

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from emlib import circuits as ci

plt.rcParams["figure.figsize"] = (6, 5)
np.set_printoptions(precision=4, suppress=True)

### A uniform strip and the sheet resistance

For a uniform sheet, $R = L/(\sigma W t)$. The combination $R_s = 1/(\sigma t)$ is the sheet resistance, quoted in ohms per square: a square of any size, $L = W$, has resistance $R_s$, and a strip has $R = R_s\,L/W$, its length measured in squares. The cell uses a resistive film with $\sigma = 10^4\,$S/m and $t = 10\,\mu$m, so $R_s = 10\,\Omega$ per square, on a grid of $h = 1\,$mm. The network result is exact for a uniform strip, since every row is the same chain of equal resistors.

In [ ]:
sigma0, t = 1e4, 1e-5            # S/m and m
h = 1e-3                          # node spacing, m
R_s = 1 / (sigma0 * t)
print(f"sheet resistance R_s = {R_s:.1f} ohm per square")
for nx, ny in ((41, 11), (81, 21), (61, 61)):
    V, Ix, Iy, I = ci.solve_sheet(np.full((nx, ny), sigma0), 1.0, 0.0, h=h, thickness=t)
    L, W = (nx - 1) * h, ny * h
    R_num, R_formula = 1.0 / I, L / (sigma0 * W * t)
    print(f"{nx:3d} x {ny:2d} nodes: L = {L * 1e3:3.0f} mm, W = {W * 1e3:3.0f} mm, {L / W:.3f} squares:  "
          f"R = {R_num:.4f} ohm,  R_s L/W = {R_formula:.4f} ohm")
    assert np.isclose(R_num, R_formula, rtol=1e-12)

### Exercise: from face currents to current density

The solver returns currents through faces, which live between nodes. Plots and Joule heating want the current density $\mathbf{J}$ at the nodes. Replace the body of `face_currents_to_J` with your own version. At each node, average the currents of the two adjacent faces along each axis and divide by the face area $h\,t$. At an electrode column there is only one adjacent $x$ face, so use it alone. At an insulating edge the missing $y$ face carries zero current, so average with zero. The check uses a uniform strip, where $J_x = \sigma V/L$ everywhere and $J_y = 0$.

In [ ]:
def face_currents_to_J(Ix, Iy, h, t):
    '''Node-centred (Jx, Jy) in A/m^2 from face currents Ix (nx-1, ny) and Iy (nx, ny-1).'''
    nx, ny = Ix.shape[0] + 1, Ix.shape[1]                      # replace from here with your own version
    Ix_pad = np.concatenate([Ix[:1], Ix, Ix[-1:]], axis=0)       # one-sided at the electrodes
    Jx = 0.5 * (Ix_pad[:-1] + Ix_pad[1:]) / (h * t)
    zero = np.zeros((nx, 1))
    Iy_pad = np.concatenate([zero, Iy, zero], axis=1)            # nothing crosses the insulating edges
    Jy = 0.5 * (Iy_pad[:, :-1] + Iy_pad[:, 1:]) / (h * t)
    return Jx, Jy


nx, ny = 41, 11
V, Ix, Iy, I = ci.solve_sheet(np.full((nx, ny), sigma0), 1.0, 0.0, h=h, thickness=t)
Jx, Jy = face_currents_to_J(Ix, Iy, h, t)
L = (nx - 1) * h
print(f"J_x from faces: min {Jx.min():.6e}, max {Jx.max():.6e} A/m^2;  sigma V / L = {sigma0 * 1.0 / L:.6e} A/m^2;  max |J_y| / J_x = {np.abs(Jy).max() / (sigma0 / L):.1e}")
assert Jx.shape == (nx, ny) and Jy.shape == (nx, ny)
assert np.allclose(Jx, sigma0 / L, rtol=1e-12) and np.allclose(Jy, 0.0, atol=1e-12 * sigma0 / L)

### A sheet with smoothly varying conductivity

Now let the conductivity vary: $\sigma = \sigma_0\,[1 + 0.8\sin(2\pi y/W')\sin(\pi x/L)]$, between $0.2\sigma_0$ and $1.8\sigma_0$, with $W'$ the distance between the outer node rows. The current takes the easy route: it bends into the high-conductivity lobe and avoids the low one. The equipotentials are no longer straight. Where $\sigma$ is high they spread apart, because a small field already drives a large current; where it is low they crowd together.

Two bounds bracket the resistance, both from the fact that removing a resistor can only raise the resistance of a network and adding a short can only lower it. Cutting every vertical face leaves independent rows in parallel: an upper bound. Shorting every column of nodes together leaves columns of faces in series, each column a set of parallel resistors: a lower bound. The solver's answer must lie between them.

In [ ]:
nx, ny = 121, 61
x, y = h * np.arange(nx), h * np.arange(ny)
X, Y = np.meshgrid(x, y, indexing="ij")
L, W = (nx - 1) * h, ny * h
sigma = sigma0 * (1 + 0.8 * np.sin(2 * np.pi * Y / ((ny - 1) * h)) * np.sin(np.pi * X / L))
V, Ix, Iy, I = ci.solve_sheet(sigma, 1.0, 0.0, h=h, thickness=t)
Jx, Jy = face_currents_to_J(Ix, Iy, h, t)

Gx = 2 * sigma[:-1] * sigma[1:] / (sigma[:-1] + sigma[1:]) * t   # x-face conductances, harmonic mean as in solve_sheet
R_cut = 1 / np.sum(1 / np.sum(1 / Gx, axis=0))          # rows in parallel, no vertical faces
R_short = np.sum(1 / np.sum(Gx, axis=1))                # columns shorted: x-faces of a column in parallel, columns in series
print(f"R = {1 / I:.4f} ohm;  lower bound (columns shorted) {R_short:.4f};  upper bound (rows cut apart) {R_cut:.4f};  uniform sheet {L / (sigma0 * W * t):.4f}")
assert R_short <= 1 / I <= R_cut

fig, ax = plt.subplots(figsize=(10, 5))
pc = ax.pcolormesh(x * 1e3, y * 1e3, (sigma / sigma0).T, cmap="Greys", shading="auto", vmin=0, vmax=2)
ax.contour(x * 1e3, y * 1e3, V.T, levels=np.linspace(0.05, 0.95, 19), colors="C0", linewidths=0.8)
ax.streamplot(x * 1e3, y * 1e3, Jx.T, Jy.T, color=np.hypot(Jx, Jy).T, cmap="autumn", density=1.4, linewidth=1)
fig.colorbar(pc, ax=ax, label="sigma / sigma0")
ax.set_aspect("equal"); ax.set_xlabel("x [mm]"); ax.set_ylabel("y [mm]")
ax.set_title("current lines (colour = |J|) and equipotentials (blue) in a sheet of varying sigma")
plt.show()

The resistance sits about 1.3 percent above the uniform value, roughly a third of the way from the lower bound to the upper. The column-shorted bound is almost exactly the uniform resistance here, because the sine averages to zero across every column, so each column's total conductance is the same as a uniform sheet's. The rows-cut bound is higher, because cutting the vertical faces forces the current in a poorly conducting row to stay in it. The real current does neither: it moves sideways only as much as the potential pushes it.

### A hole and a slit: current crowding

Remove material and the current has to go round. Two cases on the same $121\times61$ sheet: a circular hole of radius 12 mm at the centre, and a slit, a cut one node wide that runs from the bottom edge 40 mm up into the 61 mm width. The current lines squeeze through what is left, and the resistance rises. For the hole there is a classical reference point: in an infinite sheet with a uniform applied current, $J$ at the two sides of a circular hole is exactly twice the far value, so the dissipation there is four times larger. Here the hole is wide compared with the strip, and the edges squeeze the flow further.

In [ ]:
def cut_sheet(kind, nx=121, ny=61, h=1e-3):
    xx, yy = h * np.arange(nx), h * np.arange(ny)
    XX, YY = np.meshgrid(xx, yy, indexing="ij")
    s = np.full((nx, ny), sigma0)
    if kind == "hole":
        s[(XX - 0.060) ** 2 + (YY - 0.030) ** 2 <= 0.012 ** 2] = 0.0
    elif kind == "slit":
        s[(np.abs(XX - 0.060) < h / 2) & (YY <= 0.040 + 1e-12)] = 0.0
    return xx, yy, s

results = {}
fig, axes = plt.subplots(1, 2, figsize=(14, 4.2))
for ax, kind in zip(axes, ("hole", "slit")):
    xx, yy, s = cut_sheet(kind)
    Vk, Ixk, Iyk, Ik = ci.solve_sheet(s, 1.0, 0.0, h=h, thickness=t)
    R0 = (len(xx) - 1) * h / (sigma0 * len(yy) * h * t)
    results[kind] = (xx, yy, s, Vk, Ixk, Iyk, Ik, R0)
    Jxk, Jyk = face_currents_to_J(Ixk, Iyk, h, t)
    Jm = np.hypot(Jxk, Jyk)
    im = ax.pcolormesh(xx * 1e3, yy * 1e3, (Jm / (sigma0 / ((len(xx) - 1) * h))).T, cmap="magma", shading="auto", vmin=0, vmax=3)
    ax.streamplot(xx * 1e3, yy * 1e3, Jxk.T, Jyk.T, color="w", density=1.2, linewidth=0.6, arrowsize=0.6)
    ax.set_aspect("equal"); ax.set_title(f"{kind}: |J| / J_uniform, R / R_uniform = {(1 / Ik) / R0:.3f}")
    fig.colorbar(im, ax=ax, shrink=0.8)
    print(f"{kind}: R = {1 / Ik:.3f} ohm, uniform sheet {R0:.3f} ohm, ratio {(1 / Ik) / R0:.3f}")
plt.show()

### Joule heating

The power dissipated in a face is its current times the voltage across it, $P_f = I_f\,\Delta V_f$. Give half of each face's power to each of its two nodes and divide by the cell volume $h^2t$ to get a power density $p$, the discrete form of $p = \mathbf{J}\cdot\mathbf{E} = J^2/\sigma$. In a uniform strip $p_0 = \sigma_0(V/L)^2$.

The map shows where a resistive film would heat up. Around the hole the hot spots are the two sides where the current squeezes past. At the slit the heating concentrates at the tip. The tip of a cut in a 2D sheet is a singularity: near it $J$ grows like $r^{-1/2}$ with distance $r$ from the tip, so $p$ grows like $1/r$. On a grid the peak value is set by the smallest $r$ the grid resolves, about $h$, so halving $h$ should roughly double the peak. The last part of the cell checks that on a grid twice as fine. This is why cracks and sharp notches in conductors are where they burn out.

In [ ]:
def power_density(V, Ix, Iy, h, t):
    '''Node-centred Joule power density (W/m^3): half of each face's I dV to each of its nodes.'''
    Px = Ix * (V[:-1] - V[1:])
    Py = Iy * (V[:, :-1] - V[:, 1:])
    p = np.zeros_like(V)
    p[:-1] += Px / 2; p[1:] += Px / 2
    p[:, :-1] += Py / 2; p[:, 1:] += Py / 2
    return p / (h * h * t)

fig, axes = plt.subplots(1, 2, figsize=(14, 4.2))
for ax, kind in zip(axes, ("hole", "slit")):
    xx, yy, s, Vk, Ixk, Iyk, Ik, R0 = results[kind]
    p0 = sigma0 * (1.0 / ((len(xx) - 1) * h)) ** 2
    p = power_density(Vk, Ixk, Iyk, h, t)
    k = np.unravel_index(p.argmax(), p.shape)
    print(f"{kind}: peak p / p0 = {p.max() / p0:.2f} at x = {xx[k[0]] * 1e3:.0f} mm, y = {yy[k[1]] * 1e3:.0f} mm")
    im = ax.pcolormesh(xx * 1e3, yy * 1e3, np.log10(np.maximum(p / p0, 1e-3)).T, cmap="inferno", shading="auto", vmin=-2, vmax=1.5)
    ax.set_aspect("equal"); ax.set_title(f"{kind}: log10 of p / p0")
    fig.colorbar(im, ax=ax, shrink=0.8)
plt.show()

for nx_, ny_, hh in ((121, 61, 1e-3), (241, 121, 0.5e-3)):
    xx, yy, s = cut_sheet("slit", nx_, ny_, hh)
    Vk, Ixk, Iyk, Ik = ci.solve_sheet(s, 1.0, 0.0, h=hh, thickness=t)
    p0 = sigma0 * (1.0 / ((nx_ - 1) * hh)) ** 2
    R0 = (nx_ - 1) * hh / (sigma0 * ny_ * hh * t)
    print(f"slit on a grid with h = {hh * 1e3:.1f} mm: peak p / p0 = {power_density(Vk, Ixk, Iyk, hh, t).max() / p0:.1f},  R / R_uniform = {(1 / Ik) / R0:.3f}")

The hole's peak sits beside the hole, above the factor 4 of the infinite sheet because the strip edges are close. The slit's peak sits on the node just past its tip and roughly doubles when the grid is refined, as the $1/r$ singularity predicts. The resistance changes far less with refinement than the peak does, by under two percent: the total is an integral over the whole sheet, and the singular region is small. Part of that small change is geometry, since the finer slit is half as wide.

## 4. Verify

Three checks on the hole and slit solutions and on a layered sheet.

1. Kirchhoff's current law. The net current out of every node that is not on an electrode is zero to rounding error. This is the discrete $\nabla\cdot\mathbf{J} = 0$.
2. Energy balance. The total Joule power, $\sum I_f^2/G_f$ over all faces that conduct, equals the power delivered by the electrodes, $I\,(V_{\text{left}} - V_{\text{right}})$.
3. Series layers. A sheet with $\sigma_1$ for $i \le 20$ and $\sigma_2 = 4\sigma_1$ for $i \ge 21$, on 41 columns. The harmonic-mean face between columns 20 and 21 is half a cell of each material, so the discrete interface sits at $x = 20.5h$, and the closed form is two resistors in series: $R = \big(20.5h/\sigma_1 + 19.5h/\sigma_2\big)/(W t)$.

In [ ]:
def face_G(sigma, t):
    '''Face conductances with the rule solve_sheet uses: harmonic mean, zero if either node is zero.'''
    def hm(a, b):
        with np.errstate(divide="ignore", invalid="ignore"):
            return np.where((a > 0) & (b > 0), 2 * a * b / (a + b), 0.0)
    return hm(sigma[:-1], sigma[1:]) * t, hm(sigma[:, :-1], sigma[:, 1:]) * t

for kind in ("hole", "slit"):
    xx, yy, s, Vk, Ixk, Iyk, Ik, R0 = results[kind]
    net = np.zeros_like(Vk)
    net[:-1] += Ixk; net[1:] -= Ixk
    net[:, :-1] += Iyk; net[:, 1:] -= Iyk
    resid = np.abs(net[1:-1]).max() / Ik
    Gxk, Gyk = face_G(s, t)
    P_faces = np.sum(Ixk[Gxk > 0] ** 2 / Gxk[Gxk > 0]) + np.sum(Iyk[Gyk > 0] ** 2 / Gyk[Gyk > 0])
    P_terminals = Ik * (1.0 - 0.0)
    print(f"{kind}: max |net current| at a free node / I = {resid:.1e};  sum I^2/G = {P_faces:.8e} W, I V = {P_terminals:.8e} W, ratio - 1 = {P_faces / P_terminals - 1:.1e}")
    assert resid < 1e-12
    assert abs(P_faces / P_terminals - 1) < 1e-10

nx, ny = 41, 11
s = np.full((nx, ny), sigma0); s[21:] = 4 * sigma0
Vl, Ixl, Iyl, Il = ci.solve_sheet(s, 1.0, 0.0, h=h, thickness=t)
R_exact = (20.5 * h / sigma0 + 19.5 * h / (4 * sigma0)) / (ny * h * t)
print(f"series layers: R = {1 / Il:.6f} ohm, closed form {R_exact:.6f} ohm, relative error {abs(1 / Il / R_exact - 1):.1e}")
assert np.isclose(1 / Il, R_exact, rtol=1e-12)

xl = h * np.arange(nx)
x_int = 20.5 * h
V_exact = np.where(xl <= x_int, 1 - Il * xl / (sigma0 * ny * h * t), Il * (xl[-1] - xl) / (4 * sigma0 * ny * h * t))
plt.plot(xl * 1e3, Vl[:, ny // 2], "o", ms=4, label="network, middle row")
plt.plot(xl * 1e3, V_exact, "k-", lw=1, label="two resistors in series")
plt.axvline(x_int * 1e3, color="grey", ls=":", label="discrete interface, x = 20.5 h")
plt.xlabel("x [mm]"); plt.ylabel("V [V]"); plt.legend(); plt.title("the field is four times weaker where sigma is four times larger")
plt.show()

## 5. Explore and rebuild

Explore. Make a random two-phase sheet: each node conducts ($\sigma_0$) with probability $\phi$ and is an insulator ($\sigma = 0$) otherwise. Compute the effective conductivity $\sigma_{\text{eff}}/\sigma_0 = R_{\text{uniform}}/R$ for $\phi$ from 0.3 to 1 on a $60\times60$ sheet, averaging over a few random sheets per value. Below some fraction no path connects the electrodes and the current is zero; above it, the conductivity rises steeply. Where is the threshold for this node-based (site) disorder, and why does it differ from the bond lattice of notebook 04, which loses its last spanning path at $p = 1/2$? Plot the current map just above the threshold and describe where the current flows.

Rebuild. Close this notebook. In a blank editor, build the network for an $n_x\times n_y$ sheet yourself with `emlib.circuits.grid_edges`: the face conductances with the harmonic mean, the electrode columns as fixed nodes, and the call to `emlib.circuits.solve_network`. Check it against `solve_sheet` on the slit, then reproduce the energy balance.